In [1]:
# Find the project root from either the workspace or notebooks folder.
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "src/task_b").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the NLP_PROJECT_task_b folder in VS Code before running this notebook")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Prepared FEVER records from task_b_fever_baselines.ipynb (same seed and sample size), and local models only.
import pandas as pd
from task_b.schema import parse_claim_evidence
from task_b.models import embedding_similarities, model_fingerprint, nli_predictor
from task_b.verification import choose_threshold, evaluate, fever_binary_proxy, save_run, timed_predictions
SEED, VALIDATION_PER_LABEL = 42, 2000
PREPARED = PROJECT_ROOT / "outputs/task_b/fever_prepared"
VALIDATION_DIR = PREPARED / f"validation_seed{SEED}_n{VALIDATION_PER_LABEL}"
NLI_DIR = PROJECT_ROOT / "models/task_b/roberta-large-mnli"
# SMOKE_TEST = True scores only the first SMOKE_LIMIT claims per split, for debugging; never report those scores.
SMOKE_TEST, SMOKE_LIMIT = False, 20
MODE = "smoke test (not an evaluation)" if SMOKE_TEST else "full evaluation"
RUN_DIR = PROJECT_ROOT / "outputs/task_b/embedding_nli" / datetime.now(timezone.utc).strftime(
    ("smoke_run_" if SMOKE_TEST else "run_") + "%Y%m%dT%H%M%SZ")
for path in (PREPARED / "test/records.jsonl", VALIDATION_DIR / "records.jsonl"):
    assert path.is_file(), f"{path} missing: run task_b_fever_baselines.ipynb first"

In [2]:
# MiniLM: a local snapshot folder if present, otherwise the already-cached Hugging Face copy (never downloads).
EMBEDDING_DIR = PROJECT_ROOT / "models/task_b/all-MiniLM-L6-v2"
if not EMBEDDING_DIR.is_dir():
    from huggingface_hub import snapshot_download
    EMBEDDING_DIR = Path(snapshot_download("sentence-transformers/all-MiniLM-L6-v2", local_files_only=True))
print("MiniLM snapshot:", EMBEDDING_DIR)

C:\Users\ONYX\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MiniLM snapshot: C:\Users\ONYX\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2\snapshots\1110a243fdf4706b3f48f1d95db1a4f5529b4d41


In [3]:
# Read the resolved claims without changing their gold labels or evidence.
def read_records(path):
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return rows, [parse_claim_evidence(row["claim"]) for row in rows], [row["gold_label"] for row in rows]

test_rows, test_examples, test_gold = read_records(PREPARED / "test/records.jsonl")
validation_rows, validation_examples, validation_gold = read_records(VALIDATION_DIR / "records.jsonl")
eligible = {"validation": len(validation_rows), "test": len(test_rows)}
if SMOKE_TEST:
    test_rows, test_examples, test_gold = test_rows[:SMOKE_LIMIT], test_examples[:SMOKE_LIMIT], test_gold[:SMOKE_LIMIT]
    validation_rows, validation_examples, validation_gold = (validation_rows[:SMOKE_LIMIT], validation_examples[:SMOKE_LIMIT],
                                                             validation_gold[:SMOKE_LIMIT])
else:
    baseline_run = sorted((PROJECT_ROOT / "outputs/task_b/fever_baselines").glob("run_*/majority/predictions.jsonl"))[-1]
    baseline_ids = [json.loads(line)["claim_id"] for line in baseline_run.read_text(encoding="utf-8").splitlines()]
    assert baseline_ids == [row["claim"]["claim_id"] for row in test_rows], "test claims differ from the baseline run"
    print("Same test claims as baseline run", baseline_run.parts[-3])
print(f"{MODE}: scoring validation {len(validation_rows):,} of {eligible['validation']:,} and test {len(test_rows):,} of {eligible['test']:,} resolved claims")

Same test claims as baseline run run_20261022T010816Z
full evaluation: scoring validation 3,965 of 3,965 and test 13,202 of 13,202 resolved claims


In [4]:
# Embedding baseline: largest claim-evidence cosine similarity, computed once per set (batched encoding).
import time
import warnings
import sentence_transformers
import transformers
warnings.filterwarnings("ignore", message="IProgress not found")  # progress-bar widget notice
from sentence_transformers import SentenceTransformer
transformers.logging.set_verbosity_error()  # hide the routine weight-loading report
embedding_model = SentenceTransformer(str(EMBEDDING_DIR))
similarity, seconds = {}, {}
for split, examples in [("validation", validation_examples), ("test", test_examples)]:
    start = time.perf_counter()
    similarity[split] = embedding_similarities(embedding_model, examples)
    seconds[split] = time.perf_counter() - start
print("Mean seconds per claim (batched):", {split: round(seconds[split] / len(similarity[split]), 5) for split in similarity})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6083.32it/s]

Mean seconds per claim (batched): {'validation': 0.00105, 'test': 0.00071}


In [5]:
# Coverage: eligible = resolved claims in the split; skipped = left out by smoke mode; failed = no valid score.
import math
coverage = []
for split in ("validation", "test"):
    failed = sum(not math.isfinite(score) for score in similarity[split])
    coverage.append({"model": "embedding", "split": split, "eligible": eligible[split], "processed": len(similarity[split]) - failed,
                     "skipped": eligible[split] - len(similarity[split]), "failed": failed})
assert all(row["failed"] == 0 for row in coverage), "some claims have no valid similarity; fix before evaluating"
display(pd.DataFrame(coverage))

,model,split,eligible,processed,skipped,failed
0,embedding,validation,3965,3965,0,0
1,embedding,test,13202,13202,0,0


In [6]:
# Choose the cosine cutoff by validation macro-F1 only; similarity alone cannot tell support from contradiction.
GRID = [round(0.05 * step, 2) for step in range(4, 20)]
def labels_at(scores, cutoff):
    return ["SUPPORTED" if score >= cutoff else "UNSUPPORTED" for score in scores]

embedding_cutoff, embedding_search = choose_threshold(validation_gold, lambda cutoff: labels_at(similarity["validation"], cutoff), GRID)
print(f"Chosen cutoff {embedding_cutoff} | validation macro-F1:",
      " ".join(f"{row['threshold']}={row['validation_macro_f1']:.3f}" for row in embedding_search))

Chosen cutoff 0.65 | validation macro-F1: 0.2=0.349 0.25=0.368 0.3=0.390 0.35=0.429 0.4=0.471 0.45=0.504 0.5=0.534 0.55=0.562 0.6=0.592 0.65=0.600 0.7=0.578 0.75=0.525 0.8=0.463 0.85=0.394 0.9=0.354 0.95=0.334


In [7]:
# Score the test set once with the fixed 0.5 cutoff and the validation-chosen cutoff; save each run separately.
results = {}
fingerprint = model_fingerprint(EMBEDDING_DIR)
for name, cutoff in [("embedding_fixed_0.5", 0.5), ("embedding_validation", embedding_cutoff)]:
    predictions = labels_at(similarity["test"], cutoff)
    binary = [fever_binary_proxy(label) for label in predictions]
    score = evaluate(test_gold, binary, ("SUPPORTED", "CONTRADICTED"), [seconds["test"] / len(test_gold)] * len(test_gold))
    score["raw_unsupported_count"] = predictions.count("UNSUPPORTED")
    score["evaluation_scope"] = f"FEVER SUPPORTS/REFUTES gold-evidence diagnostic (NEI excluded), {MODE}"
    results[name] = score
    save_run(RUN_DIR / name, {"method": name, "threshold": cutoff, "validation_search": embedding_search if name.endswith("validation") else None,
                              "validation_dir": str(VALIDATION_DIR), "model_path": str(EMBEDDING_DIR), "model_snapshot_sha256": fingerprint,
                              "library_version": sentence_transformers.__version__, "latency_note": "batched encoding; mean per claim",
                              "binary_proxy": "UNSUPPORTED -> CONTRADICTED for this diagnostic only",
                              "mode": MODE, "coverage": coverage},
             [{"claim_id": row["claim"]["claim_id"], "gold": row["gold_label"], "similarity": round(s, 6), "raw_prediction": p,
               "binary_prediction": b} for row, s, p, b in zip(test_rows, similarity["test"], predictions, binary)], score)

In [8]:
# NLI (evidence = premise, claim = hypothesis) runs only if a local RoBERTa-MNLI snapshot exists; it is not downloaded.
if NLI_DIR.is_dir():
    import torch
    import transformers
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(str(NLI_DIR), local_files_only=True)
    nli_model = AutoModelForSequenceClassification.from_pretrained(str(NLI_DIR), local_files_only=True).to("cuda" if torch.cuda.is_available() else "cpu")
    predictions, latency = timed_predictions(test_examples, nli_predictor(tokenizer, nli_model, torch))
    binary = [fever_binary_proxy(label) for label in predictions]
    score = evaluate(test_gold, binary, ("SUPPORTED", "CONTRADICTED"), latency)
    score["raw_unsupported_count"] = predictions.count("UNSUPPORTED")
    results["nli"] = score
    coverage.append({"model": "nli", "split": "test", "eligible": eligible["test"], "processed": len(predictions),
                     "skipped": eligible["test"] - len(predictions), "failed": 0})
    save_run(RUN_DIR / "nli", {"method": "nli", "model_path": str(NLI_DIR), "model_snapshot_sha256": model_fingerprint(NLI_DIR),
                               "library_version": transformers.__version__, "aggregation": "project-specific per-sentence rule", "mode": MODE,
                               "binary_proxy": "UNSUPPORTED -> CONTRADICTED for this diagnostic only"},
             [{"claim_id": row["claim"]["claim_id"], "gold": row["gold_label"], "raw_prediction": p, "binary_prediction": b,
               "latency_seconds": t} for row, p, b, t in zip(test_rows, predictions, binary, latency)], score)
else:
    print(f"NLI skipped: no local model at {NLI_DIR.relative_to(PROJECT_ROOT)} (download deferred).")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10615.69it/s]

In [9]:
# Mode and coverage first, then test scores, per-class scores and confusion matrices (saved under RUN_DIR).
print(MODE.upper(), "| test claims scored:", {row["model"]: row["processed"] for row in coverage if row["split"] == "test"})
display(pd.DataFrame([{"method": name, "n": s["n"], "accuracy": s["accuracy"], "macro_f1": s["macro_f1"],
                       "wrong_supported": s["incorrect_supported"], "raw_unsupported": s["raw_unsupported_count"],
                       "mean_ms": s["latency_mean_ms"]} for name, s in results.items()]).round(3))
for name, s in results.items():
    print(name)
    display(pd.DataFrame(s["per_class"]).T.round(3))
    display(pd.DataFrame(s["confusion_matrix"], index=[f"gold {l}" for l in s["labels"]], columns=[f"pred {l}" for l in s["labels"]]))
print("Saved to", RUN_DIR.relative_to(PROJECT_ROOT))

FULL EVALUATION | test claims scored: {'embedding': 13202, 'nli': 13202}


,method,n,accuracy,macro_f1,wrong_supported,raw_unsupported,mean_ms
0,embedding_fixed_0.5,13202,0.557,0.523,4697,3053,0.708
1,embedding_validation,13202,0.602,0.601,2285,7286,0.708
2,nli,13202,0.814,0.811,297,2147,341.346


embedding_fixed_0.5


,precision,recall,f1,support
SUPPORTED,0.537,0.826,0.651,6597.0
CONTRADICTED,0.625,0.289,0.395,6605.0


,pred SUPPORTED,pred CONTRADICTED
gold SUPPORTED,5452,1145
gold CONTRADICTED,4697,1908


...

,pred SUPPORTED,pred CONTRADICTED
gold SUPPORTED,3631,2966
gold CONTRADICTED,2285,4320


nli


,precision,recall,f1,support
SUPPORTED,0.937,0.674,0.784,6597.0
CONTRADICTED,0.746,0.955,0.837,6605.0


,pred SUPPORTED,pred CONTRADICTED
gold SUPPORTED,4445,2152
gold CONTRADICTED,297,6308


Saved to outputs\task_b\embedding_nli\run_20261022T015303Z
